# 04 — Modeling & Hyperparameter Tuning

**Split strategy:** 80% train / 20% held-out test. Hyperparameter search uses 5-fold cross-validation *within* the training set (this is our validation layer) — the test set is touched exactly once, at the very end, for final evaluation. This avoids the common mistake of tuning against the test set.

**Model families:**
1. **Ridge Regression** — linear family, regularized to handle the residual multicollinearity
2. **Random Forest** — bagging ensemble of trees
3. **Gradient Boosting** — boosting ensemble of trees

All preprocessing that needs to be fit (scaling) is wrapped in an `sklearn.Pipeline` so it's refit correctly inside every CV fold — no leakage.

In [1]:
import pandas as pd
import numpy as np
import joblib
import sys
sys.path.insert(0, '../src')
from utils import regression_report

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

df = pd.read_csv('../data/processed/life_expectancy_features.csv')
X = df.drop(columns=['life_expectancy', 'country'])
y = df['life_expectancy']
country = df['country']  # ID only, not a feature
print('X:', X.shape, ' y:', y.shape)

X: (2928, 17)  y: (2928,)


In [2]:
X_train, X_test, y_train, y_test, country_train, country_test = train_test_split(
    X, y, country, test_size=0.2, random_state=42)
print('Train:', X_train.shape, ' Test:', X_test.shape)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

Train: (2342, 17)  Test: (586, 17)


## Model 1 — Ridge Regression (grid search)
Search space: `alpha` (regularization strength) over a log-spaced grid — small alpha behaves close to plain linear regression, large alpha shrinks coefficients more aggressively.

In [3]:
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])
ridge_grid = {'model__alpha': [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0]}

ridge_search = GridSearchCV(ridge_pipe, ridge_grid, cv=5,
                             scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_search.fit(X_train, y_train)

print('Best alpha:', ridge_search.best_params_)
print('Best CV RMSE:', -ridge_search.best_score_)

Best alpha: {'model__alpha': 50.0}
Best CV RMSE: 4.022668264297975


In [4]:
cv_results_ridge = pd.DataFrame(ridge_search.cv_results_)[
    ['param_model__alpha', 'mean_test_score', 'std_test_score']]
cv_results_ridge['mean_test_score'] = -cv_results_ridge['mean_test_score']
cv_results_ridge.columns = ['alpha', 'mean_cv_RMSE', 'std_cv_RMSE']
cv_results_ridge.sort_values('alpha')

,alpha,mean_cv_RMSE,std_cv_RMSE
0,0.01,4.023332,0.092618
1,0.10,4.023328,0.092611
2,1.00,4.023292,0.092546
3,5.00,4.023142,0.092258
4,10.00,4.022982,0.091902
5,50.00,4.022668,0.089236
6,100.00,4.024348,0.086332


## Model 2 — Random Forest (randomized search)
Search space: `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`. Random search (12 iterations, 3-fold CV) instead of exhaustive grid, since the space is large — this environment runs single-threaded, so the search is kept small enough to complete in reasonable time while still covering the space broadly.

In [5]:
cv3 = KFold(n_splits=3, shuffle=True, random_state=42)
rf_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 8, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2', None],
}
rf_pipe = Pipeline([('model', RandomForestRegressor(random_state=42))])

rf_search = RandomizedSearchCV(rf_pipe, rf_grid, n_iter=12, cv=cv3,
                                scoring='neg_root_mean_squared_error',
                                random_state=42, n_jobs=1)
rf_search.fit(X_train, y_train)

print('Best params:', rf_search.best_params_)
print('Best CV RMSE:', -rf_search.best_score_)

Best params: {'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': None, 'model__max_depth': 15}
Best CV RMSE: 2.1075274895646836


## Model 3 — Gradient Boosting (randomized search)
Search space: `n_estimators`, `learning_rate`, `max_depth`, `subsample` (12 iterations, 3-fold CV).

In [6]:
gb_grid = {
    'model__n_estimators': [100, 150, 200],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth': [2, 3, 4],
    'model__subsample': [0.7, 0.85, 1.0],
}
gb_pipe = Pipeline([('model', GradientBoostingRegressor(random_state=42))])

gb_search = RandomizedSearchCV(gb_pipe, gb_grid, n_iter=12, cv=cv3,
                                scoring='neg_root_mean_squared_error',
                                random_state=42, n_jobs=1)
gb_search.fit(X_train, y_train)

print('Best params:', gb_search.best_params_)
print('Best CV RMSE:', -gb_search.best_score_)

Best params: {'model__subsample': 0.85, 'model__n_estimators': 200, 'model__max_depth': 4, 'model__learning_rate': 0.1}
Best CV RMSE: 2.014181548785678


## Final evaluation on the held-out test set
Each tuned model is evaluated exactly once here, on data none of them has seen during training or hyperparameter search.

In [7]:
results = []
predictions = {}
for name, search in [('Ridge', ridge_search), ('RandomForest', rf_search), ('GradientBoosting', gb_search)]:
    y_pred = search.predict(X_test)
    predictions[name] = y_pred
    results.append(regression_report(y_test, y_pred, label=name))

results_df = pd.DataFrame(results).set_index('model')
results_df

,RMSE,MAE,R2
model,,,
Ridge,3.918672,2.965946,0.827393
RandomForest,1.840877,1.145665,0.961908
GradientBoosting,1.872710,1.262624,0.960580


In [8]:
import joblib
joblib.dump(ridge_search.best_estimator_, '../models/ridge_best.joblib')
joblib.dump(rf_search.best_estimator_, '../models/random_forest_best.joblib')
joblib.dump(gb_search.best_estimator_, '../models/gradient_boosting_best.joblib')

# Save test set + predictions for the error-analysis notebook
test_out = X_test.copy()
test_out['country'] = country_test.values
test_out['y_true'] = y_test.values
for name, preds in predictions.items():
    test_out[f'pred_{name}'] = preds
test_out.to_csv('../data/processed/test_predictions.csv', index=False)
results_df.to_csv('../models/test_set_results.csv')
print('Saved models and predictions.')

Saved models and predictions.


## Summary
See `results_df` above for the head-to-head comparison. The tree-based ensembles (Random Forest, Gradient Boosting) are expected to outperform Ridge here since several relationships in this data (e.g. diminishing returns of GDP/schooling on life expectancy) are non-linear, which a linear model can't capture no matter how it's regularized. Full error analysis, including *where* each model goes wrong, is in notebook 05.